# Triton Ensemble (DeBERTa → XGBoost) on SageMaker

Deploy a two-stage **Triton ensemble** on Amazon SageMaker using **Inference Components**:

1. **DeBERTa-v3-base** — encodes text into 768-dim embeddings (ONNX, GPU)
2. **XGBoost** — classifies embeddings into binary labels (ONNX, CPU)

Triton's ensemble scheduler chains the two models automatically: clients send tokenized text and receive a class prediction in a single request.

We export both models to ONNX, build a 3-model Triton repository, and deploy it as an Inference Component on a SageMaker endpoint.

This notebook was tested with the `conda_python3` kernel on an Amazon SageMaker notebook instance of type `g4dn`.

## Contents

1. [Set up the environment](#Set-up-the-environment)
1. [Prepare models (ONNX export)](#Prepare-models-(ONNX-export))
1. [Prepare the Triton model repository](#Prepare-the-Triton-model-repository)
1. [Package and upload to S3](#Package-and-upload-to-S3)
1. [Deploy as Inference Component](#Deploy-as-Inference-Component)
1. [Run inference](#Run-inference)
1. [Terminate endpoint and clean up](#Terminate-endpoint-and-clean-up)

## Set up the environment

Install dependencies and configure the SageMaker session and IAM role.

In [ ]:
!pip install -U pip boto3 "sagemaker>=3" transformers sentencepiece torch onnx onnxscript
!pip install "tritonclient[http]" numpy protobuf
!pip install xgboost onnxmltools scikit-learn skl2onnx

In [ ]:
import boto3
import json
import numpy as np
import time

from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = boto3.Session()
region = sess.region_name

role = get_execution_role()

print(f"Region: {region}")
print(f"Role:   {role}")

In [ ]:
# Update the account ID from the example URL for your deployment region.
# See: https://aws.github.io/deep-learning-containers/reference/available_images/#region-availability
TRITON_ACCOUNT_ID = "763104351884"

base = "amazonaws.com.cn" if region.startswith("cn-") else "amazonaws.com"
triton_image_uri = (
    f"{TRITON_ACCOUNT_ID}.dkr.ecr.{region}.{base}"
    f"/sagemaker-tritonserver:24.09-py3"
)

print(f"Triton image URI: {triton_image_uri}")

## Prepare models (ONNX export)

Export DeBERTa-v3-base (with mean pooling) and train an XGBoost classifier, both to ONNX format.

This runs as a subprocess via `01_prepare_models.py` to avoid kernel memory issues from holding the full PyTorch model in the notebook process.

In [ ]:
MODEL_ID = "microsoft/deberta-v3-base"
MAX_SEQ_LEN = 128
HIDDEN_DIM = 768

!python workspace/export_models.py --workspace workspace --max-seq-len {MAX_SEQ_LEN} --step deberta
!python workspace/export_models.py --workspace workspace --step xgboost

## Prepare the Triton model repository

Build a 3-model Triton repository for the ensemble:

```
triton-serve-ensemble/
├── deberta/                     # Stage 1: text → embeddings (GPU)
│   ├── config.pbtxt
│   └── 1/
│       ├── model.onnx
│       └── model.onnx.data      # External weights (dynamo ONNX export)
├── xgboost_classifier/          # Stage 2: embeddings → prediction (CPU)
│   ├── config.pbtxt
│   └── 1/model.onnx
└── ensemble_model/              # Orchestrator (no model artifact)
    ├── config.pbtxt
    └── 1/
```

The `ensemble_model` uses Triton's `ensemble` platform to chain DeBERTa's `embeddings` output into XGBoost's `float_input`.

> **Note:** PyTorch's dynamo-based `torch.onnx.export` (opset 18+) saves weights to a separate `model.onnx.data` file. Both `model.onnx` and `model.onnx.data` must be included in the Triton model repository — without the weights file, Triton will fail the health check.

In [ ]:
ENSEMBLE_REPO = "triton-serve-ensemble"

!python workspace/build_triton_repo.py --workspace workspace --triton-repo {ENSEMBLE_REPO} --max-seq-len {MAX_SEQ_LEN}

## Package and upload to S3

In [ ]:
!tar -C {ENSEMBLE_REPO}/ -czf model.tar.gz deberta xgboost_classifier ensemble_model

sagemaker_session = Session(boto_session=sess)
bucket = sagemaker_session.default_bucket()
prefix = "triton-serve-ensemble"

s3_client = sess.client("s3")
s3_client.upload_file("model.tar.gz", bucket, f"{prefix}/model.tar.gz")
model_uri = f"s3://{bucket}/{prefix}/model.tar.gz"

print(f"Model artifact uploaded to: {model_uri}")

## Deploy as Inference Component

Deploy the Triton ensemble as an **Inference Component**. The `SAGEMAKER_TRITON_DEFAULT_MODEL_NAME` is set to `ensemble_model` so Triton routes incoming requests to the ensemble orchestrator, which internally calls both DeBERTa and XGBoost.

The resource requirements specify the compute allocation for this component:
- `memory` — minimum memory in MB
- `num_accelerators` — number of GPU devices (DeBERTa runs on GPU)
- `num_cpus` — number of CPU cores (XGBoost runs on CPU)
- `copies` — number of model copies to run

In [ ]:
from sagemaker.serve.model_builder import ModelBuilder, ModelServer
from sagemaker.core.inference_config import ResourceRequirements

ENSEMBLE_MODEL_NAME = "ensemble_model"

sm_model_name = "triton-ensemble-" + time.strftime("%Y-%m-%d-%H-%M-%S", time.gmtime())
endpoint_name = "triton-ensemble-ic-" + time.strftime("%Y-%m-%d-%H-%M-%S", time.gmtime())
inference_component_name = "triton-ensemble-ic-" + time.strftime("%Y-%m-%d-%H-%M-%S", time.gmtime())

builder = ModelBuilder(
    image_uri=triton_image_uri,
    s3_model_data_url=model_uri,
    role_arn=role,
    env_vars={"SAGEMAKER_TRITON_DEFAULT_MODEL_NAME": ENSEMBLE_MODEL_NAME},
    model_server=ModelServer.TRITON,
    instance_type="ml.g4dn.2xlarge",
    sagemaker_session=sagemaker_session,
)

# build() creates the SageMaker Model resource
builder.build(model_name=sm_model_name)

predictor = builder.deploy(
    endpoint_name=endpoint_name,
    inference_component_name=inference_component_name,
    initial_instance_count=1,
    instance_type="ml.g4dn.2xlarge",
    inference_config=ResourceRequirements(
        requests={
            "memory": 4096,
            "num_accelerators": 1,
            "num_cpus": 2,
            "copies": 1,
        }
    ),
    wait=False,
)

# Wait for endpoint and inference component to reach InService
sm_client = boto3.client("sagemaker")

print(f"Waiting for endpoint '{endpoint_name}' ...")
while True:
    resp = sm_client.describe_endpoint(EndpointName=endpoint_name)
    status = resp["EndpointStatus"]
    print(f"  Endpoint status: {status}")
    if status == "InService":
        break
    elif status == "Failed":
        raise RuntimeError(f"Endpoint failed: {resp.get('FailureReason')}")
    time.sleep(30)

print(f"\nWaiting for inference component '{inference_component_name}' ...")
while True:
    try:
        resp = sm_client.describe_inference_component(InferenceComponentName=inference_component_name)
        status = resp["InferenceComponentStatus"]
        print(f"  IC status: {status}")
        if status == "InService":
            break
        elif status == "Failed":
            raise RuntimeError(f"IC failed: {resp.get('FailureReason')}")
    except sm_client.exceptions.ClientError:
        print("  IC not created yet ...")
    time.sleep(30)

print(f"\nEndpoint name: {endpoint_name}")
print(f"Inference component name: {inference_component_name}")

## Run inference and benchmark

Once the endpoint is in service, we benchmark it using `04_benchmark.py`. The script:

- Performs client-side tokenization and sends JSON payloads to the Triton ensemble
- Routes requests via `InferenceComponentName` for IC-based deployments
- Reports latency statistics (min, mean, median, p90, p95, p99, max) and sample predictions
- Supports concurrent benchmarking with `--concurrency`

In [ ]:
!python workspace/run_benchmark.py \
    --endpoint-name {endpoint_name} \
    --inference-component-name {inference_component_name} \
    --max-seq-len {MAX_SEQ_LEN} \
    --warmup 5 \
    --iterations 50

## Terminate endpoint and clean up

Delete the inference component, endpoint, endpoint configuration, and model to avoid ongoing charges.

Resources are deleted in dependency order: inference component first, then endpoint, endpoint config, and finally the model.

In [ ]:
from sagemaker.core.resources import Endpoint, EndpointConfig, Model, InferenceComponent

# Delete inference component
ic = InferenceComponent.get(inference_component_name)
ic.delete()
print(f"Deleted inference component: {inference_component_name}")

# Wait for IC deletion before deleting endpoint
time.sleep(30)

# Delete endpoint
ep = Endpoint.get(endpoint_name)
ep.delete()
print(f"Deleted endpoint: {endpoint_name}")

# Wait for endpoint deletion before deleting config
time.sleep(30)

# Delete endpoint config
epc = EndpointConfig.get(ep.endpoint_config_name)
epc.delete()
print(f"Deleted endpoint config: {ep.endpoint_config_name}")

# Delete model
mdl = Model.get(sm_model_name)
mdl.delete()
print(f"Deleted model: {sm_model_name}")

print("\nCleanup complete.")

In [ ]:
# Optional: clean up local artifacts (keeps workspace/ scripts intact)
!rm -rf {ENSEMBLE_REPO}/
!rm -rf workspace/deberta workspace/xgboost
!rm -f model.tar.gz
print("Local artifacts cleaned up.")